# EDA — Knowledge Base CFP Audit Intelligence

Análisis exploratorio del corpus de actas del Consejo Federal Pesquero (1998–2024).

**Fuente**: JSONs parseados en `data/processed/json/` — 1.185 actas, 7.738 resoluciones/decisiones.

**Ejes de análisis**:
1. Volumen temporal (actas y resoluciones por año)
2. Distribución por tipo de resolución
3. Especies más mencionadas y evolución temporal
4. Empresas más frecuentes (potenciales beneficiarios)
5. Cuotas de captura otorgadas por especie y año
6. Resoluciones formales vs decisiones informales
7. Patrones de votación

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Apuntar al root del proyecto (funciona tanto desde notebooks/ como desde el root)
_here = Path().resolve()
ROOT = _here.parent if _here.name == 'notebooks' else _here
JSON_DIR = ROOT / 'data' / 'processed' / 'json'

print(f'Root: {ROOT}')
print(f'JSON dir existe: {JSON_DIR.exists()}')


In [ ]:
# ── Cargar todos los datos ────────────────────────────────────────────────────
actas_rows = []
res_rows = []

for p in sorted(JSON_DIR.rglob('*.json')):
    d = json.loads(p.read_text(encoding='utf-8'))
    year = d.get('year', 0)
    if year == 0:
        continue  # actas sin año inferido

    actas_rows.append({
        'year': year,
        'filename': d.get('filename', p.stem),
        'fecha': d.get('fecha'),
        'quorum': d.get('quorum'),
        'n_resoluciones': len(d.get('resoluciones', [])),
        'n_miembros': len(d.get('miembros_presentes', [])),
    })

    for r in d.get('resoluciones', []):
        es_formal = not r['numero'].startswith('D')
        res_rows.append({
            'year': year,
            'acta': d.get('filename', p.stem),
            'numero': r['numero'],
            'tipo': r['tipo'],
            'es_formal': es_formal,
            'votos_favor': r.get('votos_favor'),
            'votos_contra': r.get('votos_contra'),
            'quorum': r.get('quorum'),
            'especies': r.get('especies_mencionadas', []),
            'empresas': r.get('empresas_mencionadas', []),
            'cuotas': r.get('cuotas_toneladas', []),
        })

df_actas = pd.DataFrame(actas_rows)
df_res = pd.DataFrame(res_rows)

print(f'Actas cargadas: {len(df_actas):,}')
print(f'Resoluciones/decisiones: {len(df_res):,}')
print(f'Rango temporal: {df_actas.year.min()} – {df_actas.year.max()}')
df_res.head(3)

## 1. Volumen temporal

In [ ]:
by_year_actas = df_actas.groupby('year').size().reset_index(name='actas')
by_year_res = df_res.groupby('year').size().reset_index(name='resoluciones')
by_year = by_year_actas.merge(by_year_res, on='year')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Actas por año', 'Resoluciones/decisiones por año'))

fig.add_trace(go.Bar(x=by_year.year, y=by_year.actas, name='Actas', marker_color='steelblue'), row=1, col=1)
fig.add_trace(go.Bar(x=by_year.year, y=by_year.resoluciones, name='Resoluciones', marker_color='darkorange'), row=2, col=1)

fig.update_layout(height=500, title_text='Actividad del CFP por año (1998–2024)', showlegend=False)
fig.show()

## 2. Distribución por tipo de resolución

In [ ]:
tipo_counts = df_res['tipo'].value_counts().reset_index()
tipo_counts.columns = ['tipo', 'count']

fig = px.bar(tipo_counts, x='count', y='tipo', orientation='h',
             color='count', color_continuous_scale='Blues',
             title='Distribución por tipo de resolución/decisión',
             labels={'count': 'Cantidad', 'tipo': 'Tipo'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False)
fig.show()

# Evolución temporal por tipo (top 5)
top_tipos = tipo_counts.head(5)['tipo'].tolist()
df_tipo_year = df_res[df_res['tipo'].isin(top_tipos)].groupby(['year', 'tipo']).size().reset_index(name='n')

fig2 = px.line(df_tipo_year, x='year', y='n', color='tipo',
               title='Evolución temporal por tipo (top 5)',
               labels={'n': 'Cantidad', 'year': 'Año', 'tipo': 'Tipo'})
fig2.show()

## 3. Especies más mencionadas

In [ ]:
# Explotar lista de especies
df_especies = df_res[df_res['especies'].apply(len) > 0].explode('especies').copy()
df_especies['especie'] = df_especies['especies'].str.lower().str.strip()

top_especies = df_especies['especie'].value_counts().head(12).reset_index()
top_especies.columns = ['especie', 'menciones']

fig = px.bar(top_especies, x='especie', y='menciones',
             color='menciones', color_continuous_scale='Greens',
             title='Especies más mencionadas en resoluciones (1998–2024)',
             labels={'menciones': 'Menciones', 'especie': 'Especie'})
fig.update_layout(coloraxis_showscale=False)
fig.show()

# Heatmap especies x año
top10_esp = top_especies.head(8)['especie'].tolist()
df_esp_year = df_especies[df_especies['especie'].isin(top10_esp)]\
    .groupby(['year', 'especie']).size().reset_index(name='n')
pivot = df_esp_year.pivot(index='especie', columns='year', values='n').fillna(0)

fig2 = px.imshow(pivot, aspect='auto', color_continuous_scale='YlOrRd',
                 title='Intensidad de menciones por especie y año',
                 labels={'color': 'Menciones'})
fig2.show()

## 4. Empresas más frecuentes

In [ ]:
df_emp = df_res[df_res['empresas'].apply(len) > 0].explode('empresas').copy()
df_emp['empresa'] = df_emp['empresas'].str.strip()

# Filtrar ruido (muy cortos o genéricos)
df_emp = df_emp[df_emp['empresa'].str.len() > 5]

top_emp = df_emp['empresa'].value_counts().head(20).reset_index()
top_emp.columns = ['empresa', 'menciones']

fig = px.bar(top_emp, x='menciones', y='empresa', orientation='h',
             color='menciones', color_continuous_scale='Oranges',
             title='Top 20 empresas mencionadas en resoluciones',
             labels={'menciones': 'Menciones', 'empresa': 'Empresa'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False, height=600)
fig.show()

## 5. Cuotas de captura otorgadas

In [ ]:
# Explotar cuotas — solo resoluciones tipo cuota_captura
df_cuotas = df_res[(df_res['tipo'] == 'cuota_captura') & (df_res['cuotas'].apply(len) > 0)]\
    .explode('cuotas').copy()
df_cuotas = df_cuotas.rename(columns={'cuotas': 'toneladas'})
df_cuotas = df_cuotas[(df_cuotas['toneladas'] > 10) & (df_cuotas['toneladas'] < 500_000)]

print(f'Registros de cuotas: {len(df_cuotas):,}')
print(f'Total toneladas registradas: {df_cuotas.toneladas.sum():,.0f}')
print(f'Cuota promedio por resolución: {df_cuotas.toneladas.mean():,.0f} t')

# Distribución
fig = px.histogram(df_cuotas, x='toneladas', nbins=60, log_y=True,
                   title='Distribución de cuotas otorgadas (escala log)',
                   labels={'toneladas': 'Toneladas', 'count': 'Frecuencia'})
fig.show()

# Toneladas totales por año
ton_year = df_cuotas.groupby('year')['toneladas'].sum().reset_index()
fig2 = px.bar(ton_year, x='year', y='toneladas',
              title='Toneladas totales de cuotas registradas por año',
              labels={'toneladas': 'Toneladas', 'year': 'Año'})
fig2.show()

## 6. Resoluciones formales vs decisiones informales

In [ ]:
formal_year = df_res.groupby(['year', 'es_formal']).size().reset_index(name='n')
formal_year['categoria'] = formal_year['es_formal'].map({True: 'Formal (Nro Registro)', False: 'Decisión del cuerpo'})

fig = px.bar(formal_year, x='year', y='n', color='categoria',
             title='Resoluciones formales vs decisiones del cuerpo por año',
             labels={'n': 'Cantidad', 'year': 'Año', 'categoria': 'Tipo'},
             barmode='stack',
             color_discrete_map={'Formal (Nro Registro)': 'steelblue', 'Decisión del cuerpo': 'lightcoral'})
fig.show()

# Ratio formales por año
ratio = df_res.groupby('year')['es_formal'].mean().reset_index()
ratio.columns = ['year', 'ratio_formal']

fig2 = px.line(ratio, x='year', y='ratio_formal',
               title='Ratio de resoluciones formales por año',
               labels={'ratio_formal': 'Proporción formales', 'year': 'Año'})
fig2.add_hline(y=ratio.ratio_formal.mean(), line_dash='dash', line_color='gray',
               annotation_text=f'Promedio: {ratio.ratio_formal.mean():.1%}')
fig2.update_yaxes(tickformat='.0%')
fig2.show()

## 7. Patrones de votación

In [ ]:
df_votos = df_res[df_res['votos_favor'].notna()].copy()
df_votos['votos_contra'] = df_votos['votos_contra'].fillna(0)
df_votos['unanime'] = df_votos['votos_contra'] == 0

print(f'Resoluciones con datos de votación: {len(df_votos):,}')
print(f'Unánimes: {df_votos.unanime.sum():,} ({df_votos.unanime.mean():.1%})')
print(f'Con dissenso: {(~df_votos.unanime).sum():,}')

# Unanimidad por año
unanime_year = df_votos.groupby('year')['unanime'].agg(['sum', 'count']).reset_index()
unanime_year['ratio'] = unanime_year['sum'] / unanime_year['count']

fig = px.line(unanime_year[unanime_year['count'] >= 3], x='year', y='ratio',
              title='Tasa de unanimidad en votaciones por año',
              labels={'ratio': 'Tasa de unanimidad', 'year': 'Año'})
fig.update_yaxes(tickformat='.0%')
fig.add_hline(y=unanime_year.ratio.mean(), line_dash='dash', line_color='gray',
               annotation_text=f'Promedio: {unanime_year.ratio.mean():.1%}')
fig.show()

# Distribución de votos en contra
dissenso = df_votos[df_votos['votos_contra'] > 0]
if len(dissenso) > 0:
    fig2 = px.histogram(dissenso, x='votos_contra', nbins=10,
                        title=f'Distribución de votos en contra (n={len(dissenso)})',
                        labels={'votos_contra': 'Votos en contra', 'count': 'Frecuencia'})
    fig2.show()
else:
    print('No se encontraron resoluciones con votos en contra en el dataset actual.')

## 8. Concentración de beneficiarios (HHI aproximado)
El índice Herfindahl-Hirschman (HHI) mide concentración de mercado. >2500 = alta concentración.

In [ ]:
import numpy as np

# HHI de menciones de empresas por año
hhi_rows = []
for year, grp in df_emp.groupby('year'):
    counts = grp['empresa'].value_counts()
    total = counts.sum()
    if total < 5:
        continue
    shares = counts / total
    hhi = (shares ** 2).sum() * 10000
    hhi_rows.append({'year': year, 'hhi': hhi, 'n_empresas': len(counts)})

df_hhi = pd.DataFrame(hhi_rows)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('HHI de concentración de empresas por año',
                                    'Nº de empresas distintas mencionadas por año'))

fig.add_trace(go.Scatter(x=df_hhi.year, y=df_hhi.hhi, mode='lines+markers',
                         line=dict(color='crimson')), row=1, col=1)
fig.add_hline(y=2500, line_dash='dash', line_color='orange',
              annotation_text='Alta concentración (HHI=2500)', row=1, col=1)

fig.add_trace(go.Bar(x=df_hhi.year, y=df_hhi.n_empresas,
                     marker_color='teal'), row=2, col=1)

fig.update_layout(height=600, showlegend=False,
                  title_text='Concentración de beneficiarios (empresas mencionadas en resoluciones)')
fig.show()

print(f'Años con HHI > 2500 (alta concentración): {(df_hhi.hhi > 2500).sum()}')
print(f'HHI promedio: {df_hhi.hhi.mean():.0f}')

## 9. Resumen ejecutivo del corpus

In [ ]:
print('=' * 60)
print('RESUMEN EJECUTIVO — KB CFP AUDIT INTELLIGENCE')
print('=' * 60)
print(f'Período cubierto:         {df_actas.year.min()} – {df_actas.year.max()}')
print(f'Total actas analizadas:   {len(df_actas):,}')
print(f'Total resoluciones:       {len(df_res):,}')
print(f'  - Formales (con Nro):   {df_res.es_formal.sum():,} ({df_res.es_formal.mean():.1%})')
print(f'  - Decisiones del cuerpo:{(~df_res.es_formal).sum():,} ({(~df_res.es_formal).mean():.1%})')
print(f'Tipo más frecuente:       {df_res.tipo.value_counts().index[0]} ({df_res.tipo.value_counts().iloc[0]:,})')
print(f'Especie más mencionada:   {df_especies.especie.value_counts().index[0]} ({df_especies.especie.value_counts().iloc[0]:,} menciones)')
print(f'Empresa más mencionada:   {df_emp.empresa.value_counts().index[0]} ({df_emp.empresa.value_counts().iloc[0]:,} menciones)')
print(f'Cuotas con datos:         {len(df_cuotas):,} registros')
print(f'Total toneladas reg.:     {df_cuotas.toneladas.sum():,.0f} t')
if len(df_votos) > 0:
    print(f'Tasa de unanimidad:       {df_votos.unanime.mean():.1%}')
print('=' * 60)